In [ ]:
import os
import random
import sys

import networkx as nx
import numpy as np
import pandas as pd
from scipy.linalg import eigh, orthogonal_procrustes
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from cellmaps_ppi_embedding.runner import Node2VecEmbeddingGenerator

sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import ppi_from_emb, concat_emb

### SEC

In [ ]:
def run_n2v(infile, outfile, seed=1):
    random.seed(seed)
    np.random.seed(seed)

    edges = pd.read_csv(infile, sep="\t")
    g = nx.Graph()
    g.add_edges_from(edges[["GeneA", "GeneB"]].values.tolist())

    # the main analysis's node2vec settings, from its 1.3_n2v_grid
    gen = Node2VecEmbeddingGenerator(
        nx_network=g,
        workers=1,
        dimensions=128,
        p=0.25,
        q=0.25,
        walk_length=320,
        num_walks=40,
    )
    emb = pd.DataFrame(list(gen.get_next_embedding()))
    emb.to_csv(outfile, sep="\t", index=False, header=True)

In [ ]:
for context in ["Interphase", "Mitosis"]:
    run_n2v(
        f"../../result/Hela/input/SEC-{context}.ppi",
        f"../../result/Hela/input/SEC-{context}.tsv",
    )

### TPP

In [ ]:
def kernel_embedding(X, n_dim):
    """
    Compute curve score 1 / (1 + Euclidean distance) then Z-scored
    Keep top n_dim eigencomponents scaled by the square root of their eigenvalues
    """
    S = squareform(pdist(X))
    S += 1.0
    np.reciprocal(S, out=S)

    # the diagonal is exactly 1, so off-diagonal moments come from sums
    n = len(S)
    pairs = n * (n - 1)
    mean = (S.sum() - n) / pairs
    square = (np.einsum("ij,ij->", S, S) - n) / pairs
    S -= mean
    S /= np.sqrt(square - mean ** 2)
    np.fill_diagonal(S, 0.0)

    row = S.mean(axis=1, keepdims=True)
    S -= row
    S -= row.T
    S += row.mean()
    values, vectors = eigh(S, subset_by_index=[n - n_dim, n - 1])
    order = np.argsort(values)[::-1]
    return vectors[:, order] * np.sqrt(values[order].clip(min=0.0))

In [ ]:
curves = {
    context: pd.read_csv(
        f"../../result/Hela/input/TPP-{context}.data",
        sep="\t", index_col=0
    )
    for context in ["Interphase", "Mitosis"]
}
stacked = pd.concat([
    curves[context] for context in ["Interphase", "Mitosis"]
])
embedding = kernel_embedding(stacked.to_numpy(dtype=float), 128)

sec_norm = np.median(np.concatenate([
    np.linalg.norm(
        pd.read_csv(
            f"../../result/Hela/input/SEC-{context}.tsv",
            sep="\t", index_col=0
        ).to_numpy(dtype=float),
        axis=1,
    )
    for context in ["Interphase", "Mitosis"]
]))
embedding *= sec_norm / np.median(np.linalg.norm(embedding, axis=1))

tpp_emb = {}
start = 0
for context in ["Interphase", "Mitosis"]:
    stop = start + len(curves[context])
    frame = pd.DataFrame(
        embedding[start:stop],
        index=curves[context].index,
    )
    start = stop
    tpp_emb[context] = frame

    rows = [
        [gene] + list(values)
        for gene, values in zip(frame.index, frame.to_numpy())
    ]
    pd.DataFrame(rows).to_csv(
        f"../../result/Hela/input/TPP-{context}.tsv",
        sep="\t", index=False, header=True
    )

### Stats for future filtering

In [ ]:
# all possible pairs among proteins in either modality
n_pair = {}
for context in ["Interphase", "Mitosis"]:
    genes = set()
    for modality in ["SEC", "TPP"]:
        emb = pd.read_csv(
            f"../../result/Hela/input/{modality}-{context}.tsv",
            sep="\t",
            index_col=0,
        )
        genes |= set(emb.index)
    n_pair[context] = len(genes) * (len(genes) - 1) // 2

# top 1%, add other when needed
stat = pd.DataFrame(
    [
        [int(n_pair[context] * 0.01) for context in ["Interphase", "Mitosis"]]
    ],
    index=["Percent1"],
    columns=["Interphase", "Mitosis"],
)
stat.to_csv("../../result/Hela/input/pair_stat.tsv", sep="\t")

### Concatenate the modality embeddings

In [ ]:
for context in ["Interphase", "Mitosis"]:
    emb = [
        pd.read_csv(
            f"../../result/Hela/input/{modality}-{context}.tsv",
            sep="\t",
            index_col=0,
        )
        for modality in ["SEC", "TPP"]
    ]
    concat_emb(emb).to_csv(
        f"../../result/Hela/input/Concat-{context}.tsv",
        sep="\t",
        header=True,
        index=True,
    )